# Extracting subsets of proteins from L1KD dataset, creating primary pairs with ORF2p and combined sequence FASTA files
*1. FC significant negative (89)*

1.1 Extracting significant negative FC proteins (p-value < 0.05 and FC ≤ -1).
* OUTPUT: ../results/L1KD_significant_FC_neg.csv





In [3]:
import pandas as pd

df = pd.read_excel("/mnt/storage/ana01/data/13100_2024_339_MOESM1_ESM.xlsx", sheet_name=11, header=0, engine="openpyxl")

df["Significant Negative FC"] = (df["P-value adjusted"] < 0.05) & (df["Log2 Fold-Change"] <= -1)
df_significant_neg = df[df["Significant Negative FC"] == True]

print(f"Number of significant negative FC proteins: {len(df_significant_neg)}")


df_significant_neg.to_csv("/mnt/storage/ana01/results/L1KD_significant_FC_neg.csv", index=False)
print("Significant negative FC proteins saved to ...results/L1KD_significant_FC_neg.csv")



Number of significant negative FC proteins: 89
Significant negative FC proteins saved to ...results/L1KD_significant_FC_neg.csv


1.2 Creating primary pairs of significant hits with the bait ('O00370', 'ORF2').

* OUTPUT CSV: "/mnt/storage/ana01/results/L1KD_primary_pairs_sig_FC_neg.csv"
    * 2 columns; Protein A and Protein B with touples (Protein ID, Gene Name)


In [4]:
primary_pairs_df = pd.DataFrame()


primary_pairs_df['Protein A']= list(zip(df_significant_neg['Protein ID'], df_significant_neg['Gene names']))
primary_pairs_df['Protein B']= [('O00370', 'ORF2')] * len(df_significant_neg)

primary_pairs_df.to_csv("/mnt/storage/ana01/results/L1KD_primary_pairs_sig_FC_neg.csv", index=False)

print(f"{len(primary_pairs_df)} primary pairs for significant negative FC proteins saved to ...results/L1KD_primary_pairs_sig_FC_neg.csv")

89 primary pairs for significant negative FC proteins saved to ...results/L1KD_primary_pairs_sig_FC_neg.csv


1.3  Creating combined FASTA files 
* INPUT: primary pairs CSV 
* fetching sequences from UniProt via UniProt ID
* cleaning gene names --> fasta headers --> file names (so they dont disrupt the pipeline downstream)
* combining 2 sequences and headers with clean names into one FASTA file 
* OUTPUT: FASTA files written to /mnt/storage/ana01/results/fasta_files_sig_FC_neg


In [ ]:
import requests
from textwrap import wrap
import os
import pandas as pd
import ast

#Import "../results/L1KD_primary_pairs_sig_FC_neg.csv"
df = pd.read_csv("/mnt/storage/ana01/results/L1KD_primary_pairs_sig_FC_neg.csv")

#Create output directory for FASTA files
fasta_dir = "/mnt/storage/ana01/results/fasta_files_sig_FC_neg"
os.makedirs(fasta_dir, exist_ok=True)


# Function that fetches individual protein fasta from UniProt
def fetch_uniprot_fasta(acc): 
     url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
     r = requests.get(url, timeout = 30)
     r.raise_for_status()
     return r.text.strip()

# Loop over each row in the dataframe and create FASTA files for each protein pair
tracker = 0 
for i in range(df.shape[0]):
    protein_A = ast.literal_eval(df.iloc[i,0])
    protein_B = ast.literal_eval(df.iloc[i,1])   

    #Gene Name : UniProt ID dictionary 
    ACCESSIONS = { protein_A[1]: protein_A[0], protein_B[1]:protein_B[0] }
   
    # Clean gene names (e.g., MAGOH/B → MAGOHB)
    clean_names = {}
    for name, uid in ACCESSIONS.items():
        clean_name = name.replace("/", "")
        clean_names[clean_name] = uid

    # Fetch FASTA sequences and reformat
    records = []
    for name, ID in clean_names.items():
        raw_fasta = fetch_uniprot_fasta(ID)

        # Reformat 
        lines = raw_fasta.splitlines() 
        seq = "".join(line.strip() for line in lines[1:] if line and not line.startswith(">")) 

        # Use the clean name in the header
        new_header = f">{name}" 
        seq_wrapped = "\n".join(wrap(seq, 60)) 
        records.append(f"{new_header}\n{seq_wrapped}")

    combined_fasta = "\n".join(records) 

    # Write to disk
    keys = list(clean_names.keys())
    fasta_path = os.path.join(fasta_dir, f"{keys[0]}_{keys[1]}_multimer.fasta")

    with open(fasta_path, "w") as f:
        f.write(combined_fasta) 
    
    tracker += 1
    
print(f"{tracker} FASTA files written to: {fasta_dir}\n")


89 FASTA files written to: /mnt/storage/ana01/results/fasta_files_sig_FC_neg



*2. FC significant positive (56)*

2.1 Extraction of the significant FC positive subset

In [9]:
import pandas as pd

df = pd.read_excel("/mnt/storage/ana01/data/13100_2024_339_MOESM1_ESM.xlsx", sheet_name=11, header=0, engine="openpyxl")

df["Significant Positive FC"] = (df["P-value adjusted"] < 0.05) & (df["Log2 Fold-Change"] >= 1)
df_significant_pos = df[df["Significant Positive FC"] == True]

print(f"Number of significant positive FC proteins: {len(df_significant_pos)}")


df_significant_pos.to_csv("/mnt/storage/ana01/results/L1KD_significant_FC_pos.csv", index=False)
print("Significant positive FC proteins saved to ...results/L1KD_significant_FC_pos.csv")


Number of significant positive FC proteins: 56
Significant positive FC proteins saved to ...results/L1KD_significant_FC_pos.csv


2.2 Primary pairs

In [10]:
primary_pairs_df = pd.DataFrame()


primary_pairs_df['Protein A']= list(zip(df_significant_pos['Protein ID'], df_significant_pos['Gene names']))
primary_pairs_df['Protein B']= [('O00370', 'ORF2')] * len(df_significant_pos)

primary_pairs_df.to_csv("/mnt/storage/ana01/results/L1KD_primary_pairs_sig_FC_pos.csv", index=False)

print(f"{len(primary_pairs_df)} primary pairs for significant positive FC proteins saved to ...results/L1KD_primary_pairs_sig_FC_pos.csv")

56 primary pairs for significant positive FC proteins saved to ...results/L1KD_primary_pairs_sig_FC_pos.csv


2.3 Fasta files

In [11]:
import requests
from textwrap import wrap
import os
import pandas as pd
import ast

#Import "../results/L1KD_primary_pairs_sig_FC_pos.csv"
df = pd.read_csv("/mnt/storage/ana01/results/L1KD_primary_pairs_sig_FC_pos.csv")

#Create output directory for FASTA files
fasta_dir = "/mnt/storage/ana01/results/fasta_files_sig_FC_pos"
os.makedirs(fasta_dir, exist_ok=True)


# Function that fetches individual protein fasta from UniProt
def fetch_uniprot_fasta(acc): 
     url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"
     r = requests.get(url, timeout = 30)
     r.raise_for_status()
     return r.text.strip()

# Loop over each row in the dataframe and create FASTA files for each protein pair
tracker = 0 
for i in range(df.shape[0]):
    protein_A = ast.literal_eval(df.iloc[i,0])
    protein_B = ast.literal_eval(df.iloc[i,1])   

    #Gene Name : UniProt ID dictionary 
    ACCESSIONS = { protein_A[1]: protein_A[0], protein_B[1]:protein_B[0] }
   
    # Clean gene names (e.g., MAGOH/B → MAGOHB)
    clean_names = {}
    for name, uid in ACCESSIONS.items():
        clean_name = name.replace("/", "")
        clean_names[clean_name] = uid

    # Fetch FASTA sequences and reformat
    records = []
    for name, ID in clean_names.items():
        raw_fasta = fetch_uniprot_fasta(ID)

        # Reformat 
        lines = raw_fasta.splitlines() 
        seq = "".join(line.strip() for line in lines[1:] if line and not line.startswith(">")) 

        # Use the clean name in the header
        new_header = f">{name}" 
        seq_wrapped = "\n".join(wrap(seq, 60)) 
        records.append(f"{new_header}\n{seq_wrapped}")

    combined_fasta = "\n".join(records) 

    # Write to disk
    keys = list(clean_names.keys())
    fasta_path = os.path.join(fasta_dir, f"{keys[0]}_{keys[1]}_multimer.fasta")

    with open(fasta_path, "w") as f:
        f.write(combined_fasta) 
    
    tracker += 1
    
print(f"{tracker} FASTA files written to: {fasta_dir}\n")


56 FASTA files written to: /mnt/storage/ana01/results/fasta_files_sig_FC_pos

